## 1. Setup & Imports

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.components.data_ingestion import DataIngestion
from src.components.data_preprocessing import LogPreprocessor

## 2. Load Raw Data

In [2]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.components.data_ingestion import DataIngestion

ingestion = DataIngestion()
ingestion.start_spark_session()

df = ingestion.load_data("data/raw/HDFS.log")

df.show(5, truncate=False)
df.printSchema()
print("Total rows:", df.count())

Resolved path: c:\Users\Priyanshu\Desktop\Main\ML\Distributed-Log-Analysis\data\raw\HDFS.log
+----------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                     |
+----------------------------------------------------------------------------------------------------------------------------------------------------------+
|081109 203518 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010            |
|081109 203518 35 INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: /mnt/hadoop/mapred/system/job_200811092030_0001/job.jar. blk_-1608999687919862906|
|081109 203519 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.10.6:40524 dest: /

## 3. Parse Logs (CORE STEP)

In [3]:
ingestion = DataIngestion()
ingestion.start_spark_session()

df = ingestion.load_data("data/raw/HDFS.log")

print("Raw Data Count:", df.count())
df.show(5, truncate=False)

Resolved path: c:\Users\Priyanshu\Desktop\Main\ML\Distributed-Log-Analysis\data\raw\HDFS.log
Raw Data Count: 11175629
+----------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                     |
+----------------------------------------------------------------------------------------------------------------------------------------------------------+
|081109 203518 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010            |
|081109 203518 35 INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: /mnt/hadoop/mapred/system/job_200811092030_0001/job.jar. blk_-1608999687919862906|
|081109 203519 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /

In [4]:
preprocessor = LogPreprocessor()

parsed_df = preprocessor.parse_logs(df)

parsed_df.show(5, truncate=False)
parsed_df.printSchema()

+------+------+----------+---------+----------------------------+------------------------------------------------------------------------------------------------------------------+
|date  |time  |process_id|log_level|component                   |message                                                                                                           |
+------+------+----------+---------+----------------------------+------------------------------------------------------------------------------------------------------------------+
|081109|203518|143       |INFO     |dfs.DataNode$DataXceiver    |Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010                     |
|081109|203518|35        |INFO     |dfs.FSNamesystem            |BLOCK* NameSystem.allocateBlock: /mnt/hadoop/mapred/system/job_200811092030_0001/job.jar. blk_-1608999687919862906|
|081109|203519|143       |INFO     |dfs.DataNode$DataXceiver    |Receiving block blk_-160899968

## 4. Basic Validation

### Check log levels

In [5]:
parsed_df.select("log_level").distinct().show()

+---------+
|log_level|
+---------+
|     INFO|
|     WARN|
+---------+



### Count each log level

In [6]:
parsed_df.groupBy("log_level").count().show()

+---------+--------+
|log_level|   count|
+---------+--------+
|     INFO|10812836|
|     WARN|  362793|
+---------+--------+



### Check for missing values

In [8]:
from pyspark.sql.functions import col

parsed_df.select([
    col(c).isNull().alias(c) for c in parsed_df.columns
]).show(5)

+-----+-----+----------+---------+---------+-------+
| date| time|process_id|log_level|component|message|
+-----+-----+----------+---------+---------+-------+
|false|false|     false|    false|    false|  false|
|false|false|     false|    false|    false|  false|
|false|false|     false|    false|    false|  false|
|false|false|     false|    false|    false|  false|
|false|false|     false|    false|    false|  false|
+-----+-----+----------+---------+---------+-------+
only showing top 5 rows



### Count nulls per column

In [9]:
from pyspark.sql.functions import sum as spark_sum

parsed_df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in parsed_df.columns
]).show()

+----+----+----------+---------+---------+-------+
|date|time|process_id|log_level|component|message|
+----+----+----------+---------+---------+-------+
|   0|   0|         0|        0|        0|      0|
+----+----+----------+---------+---------+-------+



## 5. Detect Invalid / Corrupted Rows

In [10]:
parsed_df.filter(
    (parsed_df.date == "") |
    (parsed_df.time == "") |
    (parsed_df.log_level == "")
).show(5, truncate=False)

+----+----+----------+---------+---------+-------+
|date|time|process_id|log_level|component|message|
+----+----+----------+---------+---------+-------+
+----+----+----------+---------+---------+-------+



## 6. Clean the Dataset

In [11]:
clean_df = parsed_df.filter(
    (parsed_df.date != "") &
    (parsed_df.time != "") &
    (parsed_df.log_level != "")
)

### Verify cleaning

In [12]:
print("Before cleaning:", parsed_df.count())
print("After cleaning:", clean_df.count())

Before cleaning: 11175629
After cleaning: 11175629


## 7. Data Type Fix (IMPORTANT)

In [13]:
from pyspark.sql.functions import col

clean_df = clean_df.withColumn("process_id", col("process_id").cast("int"))

## 8. Create Timestamp Column (VERY IMPORTANT FOR ANALYSIS)

In [14]:
from pyspark.sql.functions import concat_ws, to_timestamp

clean_df = clean_df.withColumn(
    "datetime",
    to_timestamp(concat_ws(" ", clean_df.date, clean_df.time), "yyMMdd HHmmss")
)

In [15]:
clean_df.select("date", "time", "datetime").show(5)

+------+------+-------------------+
|  date|  time|           datetime|
+------+------+-------------------+
|081109|203518|2008-11-09 20:35:18|
|081109|203518|2008-11-09 20:35:18|
|081109|203519|2008-11-09 20:35:19|
|081109|203519|2008-11-09 20:35:19|
|081109|203519|2008-11-09 20:35:19|
+------+------+-------------------+
only showing top 5 rows



## 9. Final Clean Dataset Preview

In [16]:
clean_df.show(5, truncate=False)
clean_df.printSchema()

+------+------+----------+---------+----------------------------+------------------------------------------------------------------------------------------------------------------+-------------------+
|date  |time  |process_id|log_level|component                   |message                                                                                                           |datetime           |
+------+------+----------+---------+----------------------------+------------------------------------------------------------------------------------------------------------------+-------------------+
|081109|203518|143       |INFO     |dfs.DataNode$DataXceiver    |Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010                     |2008-11-09 20:35:18|
|081109|203518|35        |INFO     |dfs.FSNamesystem            |BLOCK* NameSystem.allocateBlock: /mnt/hadoop/mapred/system/job_200811092030_0001/job.jar. blk_-1608999687919862906|2008-11-09 20:35